In [ ]:
import os
import cv2
from ultralytics import YOLO

MODEL_PATH = 'models/weights/best.pt'
SOURCE_IMAGES = 'data/raw'
OUTPUT_CROPS = 'data/classifier_dataset'

if not os.path.exists(MODEL_PATH):
    print(f"Error: No encuentro el modelo en {MODEL_PATH}")
else:
    model = YOLO(MODEL_PATH)
    results = model.predict(source=SOURCE_IMAGES, conf=0.3, verbose=False)
    
    print(f"Procesando imágenes de {SOURCE_IMAGES}...")
    
    for i, r in enumerate(results):
        for j, box in enumerate(r.boxes):
            cls = int(box.cls)
            name = r.names[cls]
            
            # Crear carpetas por categoria de producto
            target_dir = os.path.join(OUTPUT_CROPS, name)
            os.makedirs(target_dir, exist_ok=True)
            
            # Recortar y guardar
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            crop = r.orig_img[y1:y2, x1:x2]
            
            if crop.size > 0:
                cv2.imwrite(os.path.join(target_dir, f"prod_{i}_{j}.jpg"), crop)
    
    print(f"Completado revisar la nueva carpeta: {OUTPUT_CROPS}")

In [ ]:
import torch.nn as nn
from torchvision import models

model_resnet = models.resnet50(weights='IMAGENET1K_V1')

for param in model_resnet.parameters():
    param.requires_grad = False

num_classes = 10 
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, num_classes)

print("Arquitectura ResNet-50 lista para la clasificacion de productos")